# JWM-Read v2 — anti-shortcut retrain (Kaggle T4)
v1 post-mortem: real-word curriculum let the model parrot language to 0.78 tok_acc
while never using the image (blind-control Δ=0.000). v2 fixes:

1. **Anti-shortcut curriculum** — random char strings (language-unpredictable) force
   gradient through the vision stem; real text only enters later
2. **Stage gates on FREE-RUNNING CER** (rand-text probes), not teacher-forced tok_acc
3. **EOS loss ×4** — model must learn to stop
4. **KV-cached decoding** (8.5× faster) makes mid-training eval affordable

**Settings**: Accelerator = GPU T4, Internet = ON, Persistence = Files. Use
**Save & Run All (Commit)** — survives session death, output persists.


In [ ]:
# 1) Code: clone our repo
import os, sys, subprocess
REPO = '/kaggle/working/mini-world-model'
PUB_URL = 'https://github.com/anhsown/mini-world-model'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', PUB_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
sys.path.insert(0, REPO)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.get_device_name(0))


In [ ]:
# 2) Dataset (research license — do not redistribute)
from huggingface_hub import snapshot_download
DATA = '/kaggle/tmp/vdoc'
snapshot_download('trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning',
                  repo_type='dataset', local_dir=DATA,
                  allow_patterns=['data/vdoc.jsonl', 'shards/*.tar'], max_workers=4)
print('downloaded')


In [ ]:
# 3) Extract tars, drop tars
import os
from jwm.read_data import extract_tars
IMAGES = '/kaggle/tmp/vdoc_images'
extract_tars(f'{DATA}/shards', IMAGES)
for f in os.listdir(f'{DATA}/shards'):
    if f.endswith('.tar'):
        os.remove(f'{DATA}/shards/{f}')
print('extracted; disk:', os.popen('df -h /kaggle/tmp | tail -1').read().strip())


In [ ]:
# 4) Data + gate eval sets
from jwm.configs import reader_scale
from jwm.read_data import (find_fonts, load_corpus_lines, load_doc_pairs,
                           LazyReadBatcher, PrefetchBatcher, build_eval_set)
from jwm.sdg import CameraParams

cfg = reader_scale()
cfg.eos_loss_weight = 4.0                       # Day-4 fix #3
fonts = find_fonts(); assert fonts
corpus = load_corpus_lines(f'{DATA}/data/vdoc.jsonl')
doc_pairs = load_doc_pairs(f'{DATA}/data/vdoc.jsonl', IMAGES)
cam = CameraParams(noise_std=7.2, blur_sigma=0.54, jpeg_q=55,
                   contrast=1.25, wb_shift=6.0, vignette=0.12)
eval_pairs, train_pairs = doc_pairs[:400], doc_pairs[400:]
# final eval: same seed/shape as v1 + rand probes; gate eval: rand-only, cheap
final_eval = build_eval_set(cfg, eval_pairs, fonts, corpus, cam,
                            n_synth_per_level=12, n_doc=60, n_rand_per_level=8)
gate_eval = [e for e in build_eval_set(cfg, [], fonts, corpus, cam,
             n_synth_per_level=0, n_doc=0, n_rand_per_level=8, seed=555)]
print(f'train_pairs={len(train_pairs)} final_eval={len(final_eval)} gate={len(gate_eval)}')


In [ ]:
# 5) Model + resume
import json, torch
from jwm.model import JWM

CKPT = '/kaggle/working/jwm_read2_ckpt.pt'
device = 'cuda'
model = JWM(cfg)
state = {'stage': 0, 'steps_done': 0}
if os.path.exists(CKPT):
    blob = torch.load(CKPT, map_location='cpu', weights_only=False)
    model.load_state_dict(blob['model'])
    state = blob['state']
    print('RESUMED from', state)
model = model.to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

def save_ckpt(stage, steps_done):
    torch.save({'model': model.state_dict(),
                'state': {'stage': stage, 'steps_done': steps_done},
                'cfg': cfg.__dict__}, CKPT + '.tmp')
    os.replace(CKPT + '.tmp', CKPT)


In [ ]:
# 6) Staged curriculum with FREE-RUNNING CER gates
from jwm.trainer import train_stage, eval_read

BATCH = 16
# (levels, synth_ratio, random_text_ratio, steps, lr, gate_kinds, gate_cer, max_ext)
STAGES = [
    ((1, 2),       1.0, 1.0, 2500, 3e-4,  ('randL1', 'randL2'), 0.60, 2),
    ((1, 2, 3, 4), 1.0, 0.7, 4000, 3e-4,  ('randL3',),          0.80, 1),
    ((1, 2, 3, 4), 0.6, 0.3, 5000, 1.5e-4, (),                  None, 0),
]
EXT_STEPS = 1500
history = []

def gate_cer(kinds):
    subset = [e for e in gate_eval if e['kind'] in kinds]
    r = eval_read(model, subset, cfg, device, fonts=fonts, corpus=corpus,
                  cam=cam, batch=8, log=lambda *a: None)
    return r['per_kind']['overall']['cer']

for si, (levels, sr, rr, steps, lr, gk, gc, max_ext) in enumerate(STAGES):
    if state['stage'] > si:
        print(f'stage {si}: done, skip'); continue
    done_steps = state['steps_done'] if state['stage'] == si else 0
    total = steps + max_ext * EXT_STEPS          # resume-safe upper bound
    ext_used = max(0, (done_steps - steps + EXT_STEPS - 1) // EXT_STEPS) if done_steps > steps else 0
    while True:
        target = steps + ext_used * EXT_STEPS
        remaining = target - done_steps
        if remaining > 0:
            print(f'=== stage {si}: levels={levels} synth={sr} rand={rr} '
                  f'{remaining} steps (target {target}) lr={lr} ===')
            inner = LazyReadBatcher(cfg, train_pairs if sr < 1.0 else [], fonts,
                                    corpus, cam, synth_ratio=sr, levels=levels,
                                    seed=4321 + si, random_text_ratio=rr)
            pf = PrefetchBatcher(inner, BATCH, depth=3)
            base = done_steps
            hist = train_stage(model, None, None, cfg, device, steps=remaining,
                               lr=lr, batch_size=BATCH, mode_probs={'qa': 1.0},
                               warmup=min(200, remaining // 5), log_every=50,
                               ckpt_every=500, batcher=pf,
                               ckpt_fn=lambda d, si=si, base=base: save_ckpt(si, base + d))
            pf.stop()
            history.append((si, hist))
            done_steps = target
        if not gk or gc is None:
            break
        cer = gate_cer(gk)
        print(f'  GATE stage {si}: free-running CER({gk}) = {cer:.3f} (need < {gc})')
        if cer < gc or ext_used >= max_ext:
            if cer >= gc:
                print(f'  gate NOT passed after {ext_used} extensions — moving on (logged)')
            break
        ext_used += 1
        print(f'  extending stage {si}: +{EXT_STEPS} steps ({ext_used}/{max_ext})')
    state = {'stage': si + 1, 'steps_done': 0}
    save_ckpt(si + 1, 0)
print('training complete')


In [ ]:
# 7) Final eval — same protocol as v1 (+ rand probes)
res = eval_read(model, final_eval, cfg, device, fonts=fonts, corpus=corpus,
                cam=cam, batch=8)
json.dump(res['per_kind'], open('/kaggle/working/metrics_read_v2.json', 'w'),
          indent=2, ensure_ascii=False)
for r in res['rows'][:8]:
    print(f"[{r['kind']}] CER={r['cer']:.2f}\n  ref : {r['ref'][:80]}\n  pred: {r['pred'][:80]}")


In [ ]:
# 8) Artifact + loss curve
import matplotlib.pyplot as plt
torch.save({'model': model.state_dict(), 'cfg': cfg.__dict__,
            'metrics': res['per_kind']}, '/kaggle/working/jwm_read_v2.pt')
print('saved /kaggle/working/jwm_read_v2.pt')
if history:
    plt.figure(figsize=(10, 3))
    off = 0
    for si, h in history:
        plt.plot([s + off for s in h['step']], h['loss'], lw=0.6, label=f's{si}')
        off += len(h['step'])
    plt.yscale('log'); plt.legend(); plt.title('JWM-Read v2 training loss')
    plt.tight_layout(); plt.savefig('/kaggle/working/loss_read_v2.png', dpi=110); plt.show()


## Sau khi chạy xong
Tải từ **Output**: `jwm_read_v2.pt`, `metrics_read_v2.json`, `loss_read_v2.png`
— đặt `jwm_read_v2.pt` vào `Jarvis-Vision/` rồi báo Claude chạy lại bộ benchmark
(bench_read_v1.py + blind control) để đo delta so với v1.

**Đọc gate log (cell 6)**: `GATE stage 0: CER = ...` — CER < 0.6 nghĩa là mắt đã
mở (một model học vẹt sẽ kẹt ở CER ≥ 1.0 trên chữ ngẫu nhiên, không thể gian lận).
